# 04 – Extra features and a preprocessing pipeline

Adds hip, two body ratios, ethnicity, education and income to the four MVP features, and builds one pipeline that prepares numbers and categories differently.

**Starts from** `processed/nhanes_clean.csv` (notebook 03). The MVP notebook stays unchanged so results can be compared.

**Starting point (MVP, notebook 03):** logistic regression with 4 features, test ROC-AUC 0.75.

## 4.1 Load the cleaned dataset

In [1]:
import pandas as pd

df=pd.read_csv('../data/processed/nhanes_clean.csv')


In [2]:
df.head()

,SEQN,hba1c,age_years,is_female,race_ethnicity_code,education_code,income_poverty_ratio,exam_status_code,mec_exam_weight,survey_stratum,survey_psu,weight_kg,height_cm,bmi,waist_cm,hip_cm
0,130378,5.6,43,0,6.0,5.0,5.00,2.0,54374.463898,173,2,86.9,179.5,27.0,98.3,102.9
1,130379,5.6,66,0,3.0,5.0,5.00,2.0,34084.721548,173,2,101.8,174.2,33.5,114.7,112.4
2,130380,6.2,44,1,2.0,3.0,1.41,2.0,81196.277992,174,1,69.4,152.9,29.7,93.5,98.0
3,130386,5.1,34,0,1.0,4.0,1.33,2.0,39988.452940,179,1,90.6,173.3,30.2,106.1,110.6
4,130387,5.9,68,1,3.0,5.0,1.32,2.0,20776.254850,181,1,103.5,155.9,42.6,122.0,148.9


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5767 entries, 0 to 5766
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   SEQN                  5767 non-null   int64  
 1   hba1c                 5767 non-null   float64
 2   age_years             5767 non-null   int64  
 3   is_female             5767 non-null   int64  
 4   race_ethnicity_code   5767 non-null   float64
 5   education_code        5765 non-null   float64
 6   income_poverty_ratio  5029 non-null   float64
 7   exam_status_code      5767 non-null   float64
 8   mec_exam_weight       5767 non-null   float64
 9   survey_stratum        5767 non-null   int64  
 10  survey_psu            5767 non-null   int64  
 11  weight_kg             5698 non-null   float64
 12  height_cm             5710 non-null   float64
 13  bmi                   5686 non-null   float64
 14  waist_cm              5503 non-null   float64
 15  hip_cm                5453 non-n

In [4]:
df['is_female'].value_counts()

is_female
1    3181
0    2586
Name: count, dtype: int64

## Workflow order
1. Load data
2. Create new columns in `df` (row-wise calculations only)
3. Create X (all candidate features) and y
4. Split once, stratified
5. Build the pipeline (impute, scale, encode, model)
6. Fit on train
7. Evaluate on test

## 4.2 Text labels for ethnicity and education

**Note:** row-wise recoding, so it is safe before creating X and splitting.

In [5]:
df['race_ethnicity_code']= df['race_ethnicity_code'].map({
     1: 'Mexican American',
    2: 'Other Hispanic',
    3: 'White',
    4: 'Black',
    6: 'Asian',
    7: 'Other or Multiracial'
})



In [6]:
print(df['race_ethnicity_code'].value_counts(dropna=False))

race_ethnicity_code
White                   3427
Black                    683
Other Hispanic           588
Mexican American         389
Other or Multiracial     365
Asian                    315
Name: count, dtype: int64


In [7]:
# Education labels

df['education_code']= df['education_code']. map({
    1: 'Less than 9th grade',
    2: '9th to 11th grade',
    3: 'High school',
    4: 'Some college',
    5: 'College graduate'

})

print(df['education_code'].value_counts(dropna=False))

education_code
College graduate       2077
Some college           1762
High school            1206
9th to 11th grade       446
Less than 9th grade     274
NaN                       2
Name: count, dtype: int64


## 4.3 New columns: waist-to-height and waist-to-hip ratios

In [8]:
df['waist_to_height']=df['waist_cm']/df['height_cm']
df['waist_to_hip']=df['waist_cm']/df['hip_cm']
print(df['waist_to_height'].head())
print(df['waist_to_hip'].head())

0    0.547632
1    0.658439
2    0.611511
3    0.612233
4    0.782553
Name: waist_to_height, dtype: float64
0    0.955296
1    1.020463
2    0.954082
3    0.959313
4    0.819342
Name: waist_to_hip, dtype: float64


## 4.4 Create X and y

In [9]:
# all candidate features
feature_col=['age_years', 'is_female', 'bmi', 'waist_cm', 'hip_cm', 'waist_to_height', 'waist_to_hip', 'race_ethnicity_code', 'education_code', 'income_poverty_ratio']
# select them from df (keep df[...] so X stays a DataFrame)
X=df[feature_col].copy()

In [10]:
type(X)

pandas.DataFrame

In [11]:
X.shape

(5767, 10)

In [12]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 5767 entries, 0 to 5766
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   age_years             5767 non-null   int64  
 1   is_female             5767 non-null   int64  
 2   bmi                   5686 non-null   float64
 3   waist_cm              5503 non-null   float64
 4   hip_cm                5453 non-null   float64
 5   waist_to_height       5495 non-null   float64
 6   waist_to_hip          5443 non-null   float64
 7   race_ethnicity_code   5767 non-null   str    
 8   education_code        5765 non-null   str    
 9   income_poverty_ratio  5029 non-null   float64
dtypes: float64(6), int64(2), str(2)
memory usage: 573.5 KB


In [13]:
#  to create y target 

threshold=5.7

y=(df['hba1c'] >=threshold).astype(int)

y.name='elevated_hbA1c'

print(y.value_counts())

elevated_hbA1c
0    3468
1    2299
Name: count, dtype: int64


## 4.5 Train/test split (stratified)

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(X_train.shape, X_test.shape)
print(y_test.shape, y_train.shape)

(4613, 10) (1154, 10)
(1154,) (4613,)


In [15]:
print(y_test.value_counts(normalize=True))
print(y_train.value_counts(normalize=True))

elevated_hbA1c
0    0.601386
1    0.398614
Name: proportion, dtype: float64
elevated_hbA1c
0    0.601344
1    0.398656
Name: proportion, dtype: float64


## 4.6 Baseline: dummy model

**Note:** always predicts "not elevated". Any real model must beat it.

In [16]:
from sklearn.dummy import DummyClassifier

dummy=DummyClassifier(strategy='most_frequent')

# train it on the training data
dummy.fit(X_train, y_train)

#  measure accuracy on the test data

accuracy=dummy.score(X_test, y_test)

print('dummy_accuracy:', accuracy)


dummy_accuracy: 0.6013864818024264


## 4.7 Model A2: logistic regression with the 8 numeric features

In [17]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X_train_num = X_train.select_dtypes(include='number')
X_test_num = X_test[X_train_num.columns]

model_8 = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)
model_8.fit(X_train_num, y_train)

y_pred_8 = model_8.predict(X_test_num)
print('Test accuracy:', accuracy_score(y_test, y_pred_8))


Test accuracy: 0.6854419410745234


In [18]:
from sklearn.metrics import confusion_matrix, recall_score, precision_score, roc_auc_score

# predicted probability of being elevated (class 1)
y_proba_8 = model_8.predict_proba(X_test_num)[:, 1]

print('Confusion matrix:\n', confusion_matrix(y_test, y_pred_8))
print('Recall   :', recall_score(y_test, y_pred_8))
print('Precision:', precision_score(y_test, y_pred_8))
print('ROC-AUC  :', roc_auc_score(y_test, y_proba_8))

Confusion matrix:
 [[532 162]
 [201 259]]
Recall   : 0.5630434782608695
Precision: 0.6152019002375297
ROC-AUC  : 0.7674257611828093


## 4.8 Model A: the 4 MVP features on the stratified split

**Note:** same 4 features and same pipeline as the MVP. Only the split is new.

In [19]:
cols_4 = ['age_years', 'is_female', 'bmi', 'waist_cm']
X_train_4 = X_train[cols_4]
X_test_4 = X_test[cols_4]

model_4 = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)
model_4.fit(X_train_4, y_train)

y_pred_4 = model_4.predict(X_test_4)

print('Test accuracy:', accuracy_score(y_test, y_pred_4))

Test accuracy: 0.6811091854419411


## 4.9 Metrics for Model A (4 features)

**Note:** accuracy hides which people the model misses. Recall and precision show it.

In [20]:
from sklearn.metrics import confusion_matrix, recall_score, precision_score, roc_auc_score

# predicted probability of being elevated (class 1)
y_proba_4 = model_4.predict_proba(X_test_4)[:, 1]

print('Confusion matrix:\n', confusion_matrix(y_test, y_pred_4))
print('Recall   :', recall_score(y_test, y_pred_4))
print('Precision:', precision_score(y_test, y_pred_4))
print('ROC-AUC  :', roc_auc_score(y_test, y_proba_4))

Confusion matrix:
 [[534 160]
 [208 252]]
Recall   : 0.5478260869565217
Precision: 0.6116504854368932
ROC-AUC  : 0.7484055882721463


## 4.10 Preprocessor: different preparation for numbers, education and ethnicity

In [21]:
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

num_features= X_train.select_dtypes(include='number').columns.tolist()
ordinal_feature= ['education_code']
nominal_feature=['race_ethnicity_code']


num_pipe=make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler()
)
ord_pipe=make_pipeline(
    SimpleImputer(strategy='most_frequent'),
OrdinalEncoder(categories=[['Less than 9th grade', '9th to 11th grade',
                            'High school', 'Some college', 'College graduate']]))
nom_pipe=make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OneHotEncoder(handle_unknown='ignore', sparse_output=False)
)

preprocessor= make_column_transformer(
    (num_pipe, num_features),
    (ord_pipe, ordinal_feature),
    (nom_pipe, nominal_feature)    
)
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('pipeline-1', ...), ('pipeline-2', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` a

In [22]:
preprocessor.fit_transform(X_train).shape

(4613, 15)

## 4.11 Look inside the preprocessor: names of the 15 output columns

In [23]:
print(preprocessor.get_feature_names_out())

['pipeline-1__age_years' 'pipeline-1__is_female' 'pipeline-1__bmi'
 'pipeline-1__waist_cm' 'pipeline-1__hip_cm' 'pipeline-1__waist_to_height'
 'pipeline-1__waist_to_hip' 'pipeline-1__income_poverty_ratio'
 'pipeline-2__education_code' 'pipeline-3__race_ethnicity_code_Asian'
 'pipeline-3__race_ethnicity_code_Black'
 'pipeline-3__race_ethnicity_code_Mexican American'
 'pipeline-3__race_ethnicity_code_Other Hispanic'
 'pipeline-3__race_ethnicity_code_Other or Multiracial'
 'pipeline-3__race_ethnicity_code_White']


## 4.12 Model B: logistic regression with all 10 features

In [24]:


model_b=make_pipeline(
    preprocessor,
    LogisticRegression(max_iter=1000)
)
model_b.fit(X_train, y_train)
y_pred_b=model_b.predict(X_test)

print('Test accuracy:', accuracy_score(y_test, y_pred_b))


Test accuracy: 0.7270363951473137


## 4.13 Metrics for Model B

In [25]:
from sklearn.metrics import confusion_matrix, recall_score, precision_score, roc_auc_score

# predicted probability of being elevated (class 1)
y_proba_b = model_b.predict_proba(X_test)[:, 1]

print('Confusion matrix:\n', confusion_matrix(y_test, y_pred_b))
print('Recall   :', recall_score(y_test, y_pred_b))
print('Precision:', precision_score(y_test, y_pred_b))
print('ROC-AUC  :', roc_auc_score(y_test, y_proba_b))

Confusion matrix:
 [[552 142]
 [173 287]]
Recall   : 0.6239130434782608
Precision: 0.668997668997669
ROC-AUC  : 0.7898195714822702


## 4.14 Model progress (test set, stratified split, 1,154 people: 460 elevated)

| Model | Features | Accuracy | Recall | Precision | ROC-AUC |
|---|---|---|---|---|---|
| Dummy | none | 0.601 | 0.00 | n/a | 0.50 |
| Model A | 4 numeric | 0.681 | 0.548 | 0.612 | 0.748 |
| Model A2 | 8 numeric | 0.685 | 0.563 | 0.615 | 0.767 |
| Model B | 10 (ethnicity, education added) | 0.727 | 0.624 | 0.669 | 0.790 |

**Note:** Model B uses median imputation for numbers. An earlier run with `most_frequent` gave accuracy 0.7236.

## 4.15 Save the model-ready dataset (used by notebooks 05–07)

In [26]:
model_ready = X.copy()
model_ready['elevated_hbA1c'] = y
model_ready.to_csv('../data/processed/nhanes_model_ready.csv', index=False)